## 1. Импорты и инициализация

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupShuffleSplit

import solution_utils as su
import bur_topics_utils as tu



pd.set_option("display.max_colwidth", 200)
plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
SEED = 42
BLUE, BLACK, OCHRE, GREY = "#0072CE", "#0F1418", "#C98A3C", "#9CA3AF"

## 2. Загрузка данных

`dialogs` и `messages` собираются из `hw1_train.jsonl` через `solution_utils.build_frames`.
`flags` — результаты разведочного анализа (`dialog_flags.csv`): размеры диалогов, near-dup группы, флаги шаблонов и пустых сообщений.
# Надо будет поправить, когда сливать будем, чтобы старые dialogs и messages подтягивались.

In [4]:
dialogs, messages, flags = tu.load_dialogs_and_flags() # dialogs, messages - оставим из онсовного ноутбука
print("dialogs:", dialogs.shape)
print("messages:", messages.shape)
print("flags:", flags.shape)
flags.head()

dialogs: (10000, 7)
messages: (133976, 10)
flags: (10000, 28)


,dialog_id,language,n_user_turns_declared,n_messages,n_label_turns,task_type,reaction,n_user_actual,n_chars_total,max_msg_chars,...,group_id,in_near_dup_group,unlabeled,has_empty_message,is_exact_dup,bad_roles,bad_label_count,is_template,use_topics,use_reaction
0,d094012177e,en,7,14,7,analysis,positive,7,18945,7924,...,0,False,False,False,False,False,False,False,True,True
1,d306fbe61fe,en,8,16,8,information_seeking,neutral,8,33533,5558,...,1,False,False,False,False,False,False,False,True,True
2,dd6905952c7,en,5,10,5,creation,neutral,5,15264,3789,...,2,False,False,False,False,False,False,False,True,True
3,d767a675896,en,6,12,6,information_seeking,negative,6,10368,1984,...,3,False,False,False,False,False,False,False,True,True
4,d7a6280ce8e,en,8,16,8,information_seeking,negative,8,7902,1519,...,4,False,False,False,False,False,False,False,True,True


## 3. Вспоминаем, что надо выкинуть после EDA

Фильтры для 2 задачи

- `~is_template` — первая реплика не шаблон,
- `~has_empty_message` — нет пустых реплик,
- `~in_near_dup_group` — диалог не входит в группу почти-дубликатов, иначе кластеры схлопываются.
- `use_topics` — наш флаг "испольовать для второй задачи от EDA"

In [5]:
funnel = tu.filter_funnel(flags)
display(funnel)

,шаг,осталось,доля,применяется
0,всего,10000,1.0000,True
1,~is_template,10000,1.0000,True
2,~has_empty_message,10000,1.0000,True
3,~in_near_dup_group,9702,0.9702,True
4,use_topics,9702,0.9702,True


## 4. Итоговая маска обучения

Оставляем диалоги, прошедшие все фильтры.

Вход для модели тем — первая реплика пользователя; метка `reaction`/`task_type` для тем не нужна, поэтому распределение `unlabeled` в маске близко к исходному (~70%).

In [6]:
train_mask = tu.build_train_mask(flags)
print(f"train_mask: {train_mask.sum()} из {len(flags)} ({train_mask.mean():.1%})")

train_mask: 9702 из 10000 (97.0%)


## 5. Первая реплика пользователя

Т.к. мы выкинули пустые и шаблонные запросы, то будем считать, что тему диалога можно определить по первой реплике пользователя. Обрезаем до 2000 символов: длинные выбросы (до ~20k символов) редки, но мешают TF-IDF.

In [7]:
first_user_text = tu.extract_first_user_text(messages)

## 6. Разбиение train/test по группам near-dup

Разбиение делаем по `group_id`, а не по отдельным строкам: почти-дубликаты должны попасть целиком в одну часть, иначе accuracy на тесте будет завышена.

In [8]:
train_ids, test_ids = tu.group_split(flags, train_mask, test_size=0.2, seed=SEED)
print("train:", len(train_ids), " test:", len(test_ids))

train_groups = set(flags.set_index("dialog_id").loc[train_ids, "group_id"])
test_groups = set(flags.set_index("dialog_id").loc[test_ids, "group_id"])
print("пересечение групп train/test:", len(train_groups & test_groups))

train: 7761  test: 1941
пересечение групп train/test: 0


## 7. TF-IDF со стеммингом и стоп-словами

Препроцессинг:
1. Приводим к нижнему регистру.
2. Оставляем только буквенные токены (числа и пунктуация выброшены).
3. Убираем стоп-слова NLTK для английского.
4. Применяем Snowball-стеммер.

TF-IDF с `sublinear_tf=True` — логарифмирование частоты уменьшает влияние очень частых слов.

In [9]:
texts_train_raw = first_user_text.loc[train_ids].fillna("").tolist()
texts_train = [tu.preprocess_for_topics(t) for t in texts_train_raw]

vec = TfidfVectorizer(
    min_df=3,
    max_df=0.5,
    max_features=30000,
    sublinear_tf=True,
    ngram_range=(1, 1),
)
X_train_tfidf = vec.fit_transform(texts_train)
print("TF-IDF train:", X_train_tfidf.shape)
print("первые 30 стеммированных слов:", list(vec.get_feature_names_out()[:30]))

TF-IDF train: (7761, 7683)
первые 30 стеммированных слов: ['aa', 'aaa', 'aaron', 'ab', 'abandon', 'abbey', 'abc', 'abdomen', 'abi', 'abid', 'abil', 'abl', 'abnorm', 'about', 'above', 'abroad', 'abrupt', 'absenc', 'absent', 'absolut', 'absorb', 'absorpt', 'abstract', 'abund', 'abus', 'ac', 'academ', 'academi', 'academia', 'acc']


## 8. Топ слов по сумме TF-IDF

Слова с наибольшей суммой весов на тренировочной части — это частотные запросы: `write`, `use`, `make`, `code`, `list`. По ним уже видно, какие темы преобладают (написание текстов, код, объяснения, поиск информации).

In [10]:
sums = np.asarray(X_train_tfidf.sum(axis=0)).ravel()
top_idx = np.argsort(sums)[::-1][:30]
top_words = pd.DataFrame({
    "word": vec.get_feature_names_out()[top_idx],
    "weight": sums[top_idx],
})
display(top_words)

,word,weight
0,write,157.530645
1,use,136.735915
2,make,108.004075
3,give,88.087872
4,name,81.062646
5,pleas,80.918690
6,code,75.681166
7,want,70.730480
8,like,70.464625
9,one,70.294045


## 9. Сохранение артефактов

В `artifacts/`:
- `topics_flags.csv` — флаги с `group_id`,
- `first_user_text.csv` — первые реплики пользователя с обрезкой,
- `topics_train_ids.csv` / `topics_test_ids.csv` — разбиение по группам.

TF-IDF пересчитываем на этапе кластеризации.

In [11]:
tu.save_stage1_artifacts(flags, first_user_text, train_ids, test_ids)
print("артефакты сохранены в", tu.ARTIFACTS_DIR.resolve())

артефакты сохранены в /home/pavel-ubunt/Рабочий стол/ML_mipt/MIPT_ML_homework/solution_hw1_BKZ/artifacts
